In [24]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 25 — Frame-component differential geometry (`jacopy.frame_calc`)

Companion notebook to [25_frame_calc.md](25_frame_calc.md). `jacopy.frame_calc` is jacopy's **component-level submodule** for concrete metric calculations: given a metric `g` on a frame, compute Christoffel symbols, Riemann curvature, Ricci tensor, scalar curvature, Einstein tensor — with step-by-step derivation transcripts that bridge to `ProofChain` for paper-grade LaTeX output.

Requires SymPy: `pip install "jacopy[components]"`.

## Quick taste — Schwarzschild vacuum in five lines

In [25]:
from jacopy.frame_calc import einstein_tensor, levi_civita
from jacopy.frame_calc.library import schwarzschild

F, g = schwarzschild()
G = einstein_tensor(levi_civita(g), g)
print(f'G.is_vacuum() = {G.is_vacuum()}')

G.is_vacuum() = True


## Frame setup — `CoordinateFrame`

Most physics literature uses coordinate frames (`e_a = ∂/∂x^a`). The frame's `derivative(f, a)` is `∂f/∂x^a`; `gamma(a, b, c) = 0` (coordinate frames are holonomic).

In [26]:
from jacopy.frame_calc import CoordinateFrame
import sympy as sp

t, r, theta, phi = sp.symbols('t r theta phi')
F = CoordinateFrame([t, r, theta, phi])
print(F)
print('dim:', F.dim)
print('e_r(r²) =', F.derivative(r**2, 1))
print('γ^a_bc =', F.gamma(0, 1, 0))

CoordinateFrame(name='coord(t,r,theta,phi)', dim=4)
dim: 4
e_r(r²) = 2*r
γ^a_bc = 0


## `ComponentMetric` and `inverse()`

Symmetry checked at construction. `inverse()` returns `g^{ab}` as a `(2, 0)` tensor.

In [27]:
from jacopy.frame_calc import ComponentMetric

M = sp.Symbol('M', positive=True)
g = ComponentMetric(F, sp.Matrix([
    [-(1 - 2*M/r),   0,                0,    0],
    [0,              1/(1 - 2*M/r),    0,    0],
    [0,              0,                r**2, 0],
    [0,              0,                0,    r**2 * sp.sin(theta)**2],
]))
print('g[0,0] =', g[0, 0])
print('g_inv[0,0] =', g.inverse()[0, 0])
# Verify g^{ac} g_{cb} = δ^a_b for one entry
g_inv = g.inverse()
delta_00 = sp.simplify(sum(g_inv[0, c] * g[c, 0] for c in range(4)))
print('δ^0_0 =', delta_00)

g[0,0] = 2*M/r - 1
g_inv[0,0] = r/(2*M - r)
δ^0_0 = 1


## Levi-Civita Christoffel symbols via Koszul formula

The unique torsion-free metric-compatible connection.

In [28]:
from jacopy.frame_calc import levi_civita

LC = levi_civita(g)
print(f'# non-zero Christoffel: {len(LC.nonzero_components())}')
print(f'Γ^t_tr = {LC[0, 0, 1]}')
print(f'Γ^θ_rθ = {LC[2, 1, 2]}')
print(f'Γ^r_θθ = {LC[1, 2, 2]}')

# non-zero Christoffel: 13
Γ^t_tr = M/(r*(-2*M + r))
Γ^θ_rθ = 1/r
Γ^r_θθ = 2*M - r


## Step-by-step derivation transcript

Each Christoffel computation records `KoszulStep`s. Use `format_derivation` for plain text or `derivation_chain` for `ProofChain` → LaTeX.

In [29]:
print(LC.format_derivation(0, 0, 1))

Γ^t_{tr}  (via Koszul formula)
──────────────────────────────
  [1] Koszul formula
      2 g(∇_{e_t} e_r, e_t) = e_t(g_{rc}) + e_r(g_{tc}) - e_c(g_{tr}) - γ^d_{rc} g_{dt} - γ^d_{tc} g_{dr} + γ^d_{tr} g_{dc}
  [2] Frame-derivative terms
      For each c: e_t(g_{rc}) + e_r(g_{tc}) - e_c(g_{tr})
      = (-2*M/r**2, 0, 0, 0)
  [3] γ correction (coordinate frame)
      γ^a_{bc} ≡ 0 for a holonomic frame; skipped.
  [4] Multiply by ½ g^{ec} and contract over c
      Γ^t_{tr} = ½ Σ_c g^{tc} · (frame-deriv + γ-terms)
      = -M/(r*(2*M - r))
  [5] Simplify
      sympy.simplify
      = M/(r*(-2*M + r))


## Curvature, Ricci, Einstein

Schwarzschild is Ricci-flat — `Ric = 0`, `R = 0`, `G = 0`. This is the vacuum field equation result.

In [30]:
from jacopy.frame_calc import (
    curvature, ricci, ricci_scalar, einstein_tensor,
)

R = curvature(LC)
Ric = ricci(LC)
R_scalar = ricci_scalar(LC, g)
G = einstein_tensor(LC, g)

print(f'curvature.is_zero():  {R.is_zero()}  (NOT flat)')
print(f'Ric.is_zero():        {Ric.is_zero()}')
print(f'R_scalar:             {R_scalar}')
print(f'G.is_vacuum():        {G.is_vacuum()}')

curvature.is_zero():  False  (NOT flat)
Ric.is_zero():        True
R_scalar:             0
G.is_vacuum():        True


## Optimised mode for Kerr-class metrics

Default mode runs `sympy.simplify` on every Christoffel / Ricci / curvature entry. For Kerr (off-diagonal + complex denominators), this blows up. **Optimised mode** skips per-entry simplify; expressions stay raw but mathematically correct. Trade-off: no derivation traces in optimised mode.

In [31]:
from jacopy.frame_calc.library import kerr
import time

F_kerr, g_kerr = kerr()
t0 = time.perf_counter()
G_kerr = einstein_tensor(
    levi_civita(g_kerr, optimized=True), g_kerr, optimized=True
)
elapsed = time.perf_counter() - t0
print(f'Kerr full pipeline: {elapsed:.1f} s')
print(f'G.is_vacuum(): {G_kerr.is_vacuum()}')

Kerr full pipeline: 15.7 s
G.is_vacuum(): True


## Library fixtures

Ready-made factories: `minkowski`, `schwarzschild`, `frw`, `kerr`. Each accepts `Symbol` / `Function` overrides.

In [32]:
from jacopy.frame_calc.library import minkowski, frw

# Minkowski 4D, flat
F_m, g_m = minkowski()
G_m = einstein_tensor(levi_civita(g_m), g_m)
print(f'Minkowski G.is_vacuum(): {G_m.is_vacuum()}')

# FRW (k=0, a(t) symbolic), non-vacuum cosmology
F_frw, g_frw = frw()
G_frw = einstein_tensor(levi_civita(g_frw), g_frw)
print(f'FRW G.is_zero(): {G_frw.is_zero()} (Friedmann eq form)')
print(f'FRW G[0,0] = {sp.simplify(G_frw[0, 0])}')

Minkowski G.is_vacuum(): True
FRW G.is_zero(): False (Friedmann eq form)
FRW G[0,0] = -3*Derivative(a(t), t)**2/a(t)**2


## ProofChain bridge — paper-grade LaTeX

Each tracked tensor's `derivation_chain(...)` returns a `ProofChain` compatible with `chain_to_latex_document`. The `SymPyAtom` wrapper bridges SymPy expressions into jacopy's `Expr` for ProofStep storage.

In [33]:
from jacopy.display import chain_to_latex

chain = LC.derivation_chain(0, 0, 1)   # Γ^t_tr
print(f'chain length: {len(chain.steps)}')
print(f'first step rule: {chain.steps[0].rule}')
print(f'first step tag: {chain.steps[0].provenance_tag}')

chain length: 5
first step rule: Koszul formula
first step tag: computation


## Drop-in template — paste your metric

Below is the **paper-workflow template**: change *only* the metric-matrix block; the rest of the pipeline runs as-is on whatever metric you provided. The default example uses Reissner-Nordström (charged Schwarzschild) — not in the library, just to show that any metric matrix works.

**To compute on a different metric**, edit the matrix in block 2; everything else is unchanged.

In [34]:
import sympy as sp
from jacopy.frame_calc import (
    CoordinateFrame, ComponentMetric,
    levi_civita, ricci, ricci_scalar, einstein_tensor,
)

# ─────────────────────────────────────────────────────────
# 1. Coordinates, adjust to your metric's chart
# ─────────────────────────────────────────────────────────
t, r, theta, phi = sp.symbols('t r theta phi')
coords = [t, r, theta, phi]

# Extra parameters (mass, charge, cosmological constant, …):
M = sp.Symbol('M', positive=True)
Q = sp.Symbol('Q', positive=True)

# ─────────────────────────────────────────────────────────
# 2. METRIC MATRIX, REPLACE THIS BLOCK WITH YOUR OWN
# ─────────────────────────────────────────────────────────
# Reissner-Nordström: charged static spherical black hole
factor = 1 - 2*M/r + Q**2 / r**2
metric_matrix = sp.Matrix([
    [-factor,         0,        0,                          0],
    [0,        1/factor,        0,                          0],
    [0,               0,     r**2,                          0],
    [0,               0,        0,    r**2 * sp.sin(theta)**2],
])

# ─────────────────────────────────────────────────────────
# 3. Pipeline, runs as-is on whatever metric is above
# ─────────────────────────────────────────────────────────
F = CoordinateFrame(coords)
g = ComponentMetric(F, metric_matrix)
LC = levi_civita(g)
Ric = ricci(LC)
R = ricci_scalar(LC, g)
G = einstein_tensor(LC, g)

# ─────────────────────────────────────────────────────────
# 4. Output, summary + all non-zero entries
# ─────────────────────────────────────────────────────────
names = F.index_names()
print(f'# non-zero Christoffel: {len(LC.nonzero_components())}')
print(f'Ricci scalar R   = {sp.simplify(R)}')
print(f'Ric.is_zero()    = {Ric.is_zero()}')
print(f'G.is_vacuum()    = {G.is_vacuum()}')

print('\nChristoffel symbols (non-zero):')
for (e, a, b), val in LC.nonzero_components().items():
    print(f'  Γ^{names[e]}_{{{names[a]}{names[b]}}} = {val}')

print('\nEinstein tensor (non-zero entries):')
for a in range(F.dim):
    for b in range(a, F.dim):
        val = sp.simplify(sp.trigsimp(G[a, b]))
        if val != 0:
            print(f'  G_{{{names[a]}{names[b]}}} = {val}')

# non-zero Christoffel: 13
Ricci scalar R   = 0
Ric.is_zero()    = False
G.is_vacuum()    = False

Christoffel symbols (non-zero):
  Γ^t_{tr} = (M*r - Q**2)/(r*(-2*M*r + Q**2 + r**2))
  Γ^t_{rt} = (M*r - Q**2)/(r*(-2*M*r + Q**2 + r**2))
  Γ^r_{tt} = (M*r - Q**2)*(-2*M*r + Q**2 + r**2)/r**5
  Γ^r_{rr} = (-M*r + Q**2)/(r*(-2*M*r + Q**2 + r**2))
  Γ^r_{thetatheta} = 2*M - Q**2/r - r
  Γ^r_{phiphi} = (2*M*r - Q**2 - r**2)*sin(theta)**2/r
  Γ^theta_{rtheta} = 1/r
  Γ^theta_{thetar} = 1/r
  Γ^theta_{phiphi} = -sin(2*theta)/2
  Γ^phi_{rphi} = 1/r
  Γ^phi_{thetaphi} = 1/tan(theta)
  Γ^phi_{phir} = 1/r
  Γ^phi_{phitheta} = 1/tan(theta)

Einstein tensor (non-zero entries):
  G_{tt} = Q**2*(2*M*r - Q**2 - r**2)/r**6
  G_{rr} = Q**2/(r**2*(-2*M*r + Q**2 + r**2))
  G_{thetatheta} = -Q**2/r**2
  G_{phiphi} = -Q**2*sin(theta)**2/r**2


### Other metrics you can drop in

Replace block 2 with any of these (or whatever your paper uses):

**Schwarzschild-de Sitter (cosmological constant):**
```python
Lambda = sp.Symbol('Lambda')
factor = 1 - 2*M/r - Lambda*r**2/3
metric_matrix = sp.Matrix([
    [-factor, 0, 0, 0],
    [0, 1/factor, 0, 0],
    [0, 0, r**2, 0],
    [0, 0, 0, r**2 * sp.sin(theta)**2],
])
```

**Vaidya (radiating mass, advanced time):**
```python
u = sp.Symbol('u'); coords = [u, r, theta, phi]
m = sp.Function('m')(u)
metric_matrix = sp.Matrix([
    [-(1 - 2*m/r), 1, 0, 0],
    [1, 0, 0, 0],
    [0, 0, r**2, 0],
    [0, 0, 0, r**2 * sp.sin(theta)**2],
])
```

**Kerr-class metrics** (off-diagonal, complex denominators) — add `optimized=True` to every pipeline call:
```python
LC = levi_civita(g, optimized=True)
Ric = ricci(LC, optimized=True)
R = ricci_scalar(LC, g, optimized=True)
G = einstein_tensor(LC, g, optimized=True)
```

The output is raw (un-simplified) but mathematically correct; `G.is_vacuum()` etc. still work via SymPy's basic arithmetic. Apply `sp.simplify(LC[a,b,c])` on the specific entries you want to clean up.

## Custom connection — independent of the metric

**Connection and metric are independent geometric objects.** The Levi-Civita connection is the *unique* connection that's both torsion-free and metric-compatible for a given metric — but it's just one of many possible connections. In Einstein-Cartan theory, teleparallel gravity, Palatini formulations, and other modified gravity frameworks, the connection is **not** Levi-Civita.

`einstein_tensor(connection, g)` accepts **any** `ComponentConnection`, not just `LeviCivitaConnection`. Build a custom connection with `ComponentConnection(F, christoffel_table)` and the rest of the pipeline runs as-is.

### Symbol-domain matching (important pitfall!)

When you supply Christoffel symbols by hand, **use the symbols the frame already carries** — not freshly-created ones. Library factories like `schwarzschild()` create symbols with specific assumptions (`r > 0`, `M > 0`); your hand-written `sp.symbols('r')` is a *different* symbol object even though the name matches.

Pattern:
```python
F, g = schwarzschild()
t, r, theta, phi = F.coords            # ← use these
M = sp.Symbol('M', positive=True)      # ← match factory's assumption
```

### Sanity check — manual Schwarzschild matches Levi-Civita

Build the textbook Schwarzschild Christoffels by hand, wrap them in `ComponentConnection`, and verify the result matches `levi_civita(g)` exactly.

In [35]:
from jacopy.frame_calc import (
    ComponentConnection, einstein_tensor, levi_civita,
)
from jacopy.frame_calc.library import schwarzschild
import sympy as sp

F_sw, g_sw = schwarzschild()
t_sw, r_sw, theta_sw, phi_sw = F_sw.coords
M_sw = sp.Symbol('M', positive=True)

# 13 non-zero textbook Schwarzschild Christoffels
manual = sp.MutableDenseNDimArray.zeros(F_sw.dim, F_sw.dim, F_sw.dim)
factor = 1 - 2*M_sw/r_sw
val_t = M_sw / (r_sw**2 * factor)
manual[0, 0, 1] = val_t                                          # Γ^t_tr
manual[0, 1, 0] = val_t                                          # Γ^t_rt
manual[1, 0, 0] = M_sw * factor / r_sw**2                        # Γ^r_tt
manual[1, 1, 1] = -M_sw / (r_sw**2 * factor)                     # Γ^r_rr
manual[1, 2, 2] = -(r_sw - 2*M_sw)                               # Γ^r_θθ
manual[1, 3, 3] = -(r_sw - 2*M_sw) * sp.sin(theta_sw)**2         # Γ^r_φφ
manual[2, 1, 2] = manual[2, 2, 1] = 1/r_sw                       # Γ^θ_rθ
manual[2, 3, 3] = -sp.sin(theta_sw) * sp.cos(theta_sw)           # Γ^θ_φφ
manual[3, 1, 3] = manual[3, 3, 1] = 1/r_sw                       # Γ^φ_rφ
manual[3, 2, 3] = manual[3, 3, 2] = sp.cos(theta_sw)/sp.sin(theta_sw)

manual_conn = ComponentConnection(F_sw, manual)
LC_sw = levi_civita(g_sw)

# Entry-by-entry consistency
all_match = all(
    sp.simplify(sp.trigsimp(LC_sw[a, b, c] - manual_conn[a, b, c])) == 0
    for a in range(F_sw.dim) for b in range(F_sw.dim) for c in range(F_sw.dim)
)
print(f'manual vs Levi-Civita match? {all_match}')
print(f'einstein_tensor(manual_conn, g).is_vacuum() = {einstein_tensor(manual_conn, g_sw).is_vacuum()}')
print(f'einstein_tensor(LC, g).is_vacuum()         = {einstein_tensor(LC_sw, g_sw).is_vacuum()}')

manual vs Levi-Civita match? True
einstein_tensor(manual_conn, g).is_vacuum() = True
einstein_tensor(LC, g).is_vacuum()         = True


### Non-trivial use: same metric, different connection

For modified-gravity work, you'd add a torsion correction or use a fully independent connection. Here's the same Schwarzschild metric with a torsion-perturbed connection — the Einstein tensor is no longer vacuum because the connection is no longer torsion-free.

In [36]:
from jacopy.frame_calc import torsion

# Levi-Civita baseline
LC = levi_civita(g_sw)

# Add antisymmetric torsion: T^t_{rθ} = sin θ
new_christoffel = sp.MutableDenseNDimArray.zeros(F_sw.dim, F_sw.dim, F_sw.dim)
for a in range(F_sw.dim):
    for b in range(F_sw.dim):
        for c in range(F_sw.dim):
            new_christoffel[a, b, c] = LC[a, b, c]

new_christoffel[0, 1, 2] += sp.Rational(1, 2) * sp.sin(theta_sw)
new_christoffel[0, 2, 1] -= sp.Rational(1, 2) * sp.sin(theta_sw)

torsion_conn = ComponentConnection(F_sw, new_christoffel)
T = torsion(torsion_conn)
print(f'Torsion is zero?              {T.is_zero()}')
print(f'T^t_{{rθ}} = {T[0, 1, 2]}')

G_torsion = einstein_tensor(torsion_conn, g_sw)
print(f'Same metric, custom connection: G.is_vacuum() = {G_torsion.is_vacuum()}')
print(f'Levi-Civita on same metric:     G.is_vacuum() = {einstein_tensor(LC, g_sw).is_vacuum()}')

Torsion is zero?              False
T^t_{rθ} = sin(theta)
Same metric, custom connection: G.is_vacuum() = False
Levi-Civita on same metric:     G.is_vacuum() = True


### When you'd actually use this

| Scenario | Custom connection? |
|---|---|
| Standard GR (vacuum, Einstein-Maxwell, Schwarzschild family) | No — `levi_civita(g)` |
| Einstein-Cartan theory (torsion present) | Yes |
| Teleparallel gravity (`R = 0`, torsion only) | Yes |
| Palatini formulation (`g`, `Γ` varied independently) | Yes |
| Affine theory (no metric) | Yes |

For the standard-GR cases the metric → Levi-Civita → tensors chain is all you need. The custom-connection path opens up when the physics requires it.

### API stress test — arbitrary symbols

Before showing physically-motivated patterns, let's check the API accepts completely arbitrary symbol parameters.

**Schwarzschild with made-up A, B:** the API runs, but the result is non-vacuum because A, B don't match Levi-Civita and the angular-block Christoffels are missing.

In [37]:
F_sw, g_sw = schwarzschild()
t_sw, r_sw, theta_sw, phi_sw = F_sw.coords
A, B = sp.symbols('A B')

manual = sp.MutableDenseNDimArray.zeros(F_sw.dim, F_sw.dim, F_sw.dim)
manual[0, 0, 1] = manual[0, 1, 0] = A
manual[1, 0, 0] = B
manual[1, 1, 1] = -B
manual[2, 1, 2] = manual[2, 2, 1] = 1 / r_sw
manual[3, 1, 3] = manual[3, 3, 1] = 1 / r_sw

manual_conn = ComponentConnection(F_sw, manual)
G_manual = einstein_tensor(manual_conn, g_sw)
print(f'is_vacuum?  {G_manual.is_vacuum()}')
print('non-zero G entries:')
for a in range(F_sw.dim):
    for b in range(a, F_sw.dim):
        val = sp.simplify(sp.trigsimp(G_manual[a, b]))
        if val != 0:
            print(f'  G[{a},{b}] = {val}')

is_vacuum?  False
non-zero G entries:
  G[0,0] = (B*r**2*(r*(A + B) - 2) + (2*M - r)**2*(A*r*(A + B) + 2*B))/(2*r**3)
  G[1,1] = (B*r**2*(r*(A + B) - 2) + (2*M - r)**2*(A*r*(A + B) + 2*B))/(2*r*(2*M - r)**2)
  G[2,2] = (-B*r**2*(r*(A + B) - 2) + (2*M - r)**2*(A*r*(A + B) + 2*B))/(2*(2*M - r))
  G[3,3] = (-B*r**2*(r*(A + B) - 2) + (2*M - r)**2*(A*r*(A + B) + 2*B))*sin(theta)**2/(2*(2*M - r))


**2D polar with made-up A, B:** the result will reveal that B doesn't appear in G — Lovelock 2D theorem in action!

In [38]:
x, y = sp.symbols('x y')
A, B = sp.symbols('A B')
F2 = CoordinateFrame([x, y])
g2 = ComponentMetric(F2, sp.Matrix([[1, 0], [0, x**2]]))

manual = sp.MutableDenseNDimArray.zeros(F2.dim, F2.dim, F2.dim)
manual[0, 0, 0] = A          # Γ^x_{xx}
manual[0, 1, 1] = B*x        # Γ^x_{yy}
manual[1, 0, 1] = 1/x + A    # Γ^y_{xy}
manual[1, 1, 0] = 1/x - A    # Γ^y_{yx}

manual_conn = ComponentConnection(F2, manual)
G = einstein_tensor(manual_conn, g2)
for a in range(F2.dim):
    for b in range(F2.dim):
        val = sp.simplify(sp.trigsimp(G[a, b]))
        if val != 0:
            print(f'G[{a},{b}] = {val}')
print(f'\nA=0 → vacuum?  {all(sp.simplify(G[a,b].subs(A, 0)) == 0 for a in range(2) for b in range(2))}')

G[0,0] = -A/(2*x)
G[1,1] = A*x/2

A=0 → vacuum?  True


**Lovelock 2D detected**: B doesn't appear in `G`. The torsion `T^y_{xy} = 2A` is the only thing breaking the 2D Lovelock theorem (`G ≡ 0` for any torsion-free connection in 2D). At `A = 0`, the connection becomes torsion-free regardless of B, and `G` collapses to zero.

### Physically-motivated deformation patterns

Real paper work uses one of these patterns. Each is a Levi-Civita connection plus a specific deformation tensor parameterised by a small number of physical quantities.

#### Pattern 1: Levi-Civita + antisymmetric torsion (2D polar)

Single scalar `α` controls a torsion-violating perturbation. At `α = 0` recovers Levi-Civita (vacuum, by Lovelock 2D).

In [39]:
x, y = sp.symbols('x y')
alpha = sp.Symbol('alpha', real=True)
F = CoordinateFrame([x, y])
g = ComponentMetric(F, sp.Matrix([[1, 0], [0, x**2]]))
LC = levi_civita(g)

Gamma = sp.MutableDenseNDimArray(LC.components)
Gamma[0, 0, 1] += alpha
Gamma[0, 1, 0] -= alpha
torsion_conn = ComponentConnection(F, Gamma)
G_torsion = einstein_tensor(torsion_conn, g)

print('Pattern 1: 2D polar + α torsion')
print(f'  LC vacuum?       {einstein_tensor(LC, g).is_vacuum()}')
print(f'  deformed vacuum? {G_torsion.is_vacuum()}')
for a in range(F.dim):
    for b in range(F.dim):
        val = sp.simplify(sp.trigsimp(G_torsion[a, b]))
        if val != 0:
            print(f'  G[{a},{b}] = {val}')
print(f'  α=0 recovers LC? {all(sp.simplify(G_torsion[a,b].subs(alpha, 0)) == 0 for a in range(2) for b in range(2))}')

Pattern 1: 2D polar + α torsion
  LC vacuum?       True
  deformed vacuum? False
  G[0,0] = alpha**2/(2*x**2)
  G[0,1] = -alpha/x
  G[1,0] = -alpha/x
  G[1,1] = -alpha**2/2
  α=0 recovers LC? True


#### Pattern 2: Levi-Civita + Weyl non-metricity (2D polar)

Single scalar `W_x` controls a Weyl-type deformation. **Surprise**: `G ≡ 0` even with `W_x ≠ 0`, because Weyl preserves torsion-freeness — Lovelock 2D still applies. Pedagogical lesson: torsion breaks Lovelock; non-metricity doesn't.

In [40]:
W_x = sp.Symbol('W_x', real=True)
LC = levi_civita(g)
g_mat = g.matrix()
g_inv = g_mat.inv()
W = [W_x, 0]
W_up = [sum(g_inv[a, b] * W[b] for b in range(F.dim)) for a in range(F.dim)]

Gamma = sp.MutableDenseNDimArray(LC.components)
for a in range(F.dim):
    for b in range(F.dim):
        for c in range(F.dim):
            d_ab = 1 if a == b else 0
            d_ac = 1 if a == c else 0
            Gamma[a, b, c] += sp.Rational(1, 2) * (
                d_ab * W[c] + d_ac * W[b] - g_mat[b, c] * W_up[a]
            )

weyl_conn = ComponentConnection(F, Gamma)
G_weyl = einstein_tensor(weyl_conn, g)
print('Pattern 2: 2D polar + Weyl non-metricity')
print(f'  deformed vacuum? {G_weyl.is_vacuum()}  ← Lovelock 2D, even with W_x ≠ 0')

Pattern 2: 2D polar + Weyl non-metricity
  deformed vacuum? True  ← Lovelock 2D, even with W_x ≠ 0


#### Pattern 3: Schwarzschild + antisymmetric torsion (4D)

Single scalar `ε` adds a `t-φ` cross-term torsion. The result is compact: only `G_{tφ}` and `G_{φt}` non-zero.

In [41]:
F_sw, g_sw = schwarzschild()
t, r, theta, phi = F_sw.coords
epsilon = sp.Symbol('epsilon', real=True)
LC = levi_civita(g_sw)

Gamma = sp.MutableDenseNDimArray(LC.components)
Gamma[3, 1, 0] += epsilon       # Γ^φ_{rt}
Gamma[3, 0, 1] -= epsilon       # Γ^φ_{tr}
torsion_conn = ComponentConnection(F_sw, Gamma)

G_torsion = einstein_tensor(torsion_conn, g_sw)
print('Pattern 3: Schwarzschild + ε torsion')
print(f'  deformed vacuum? {G_torsion.is_vacuum()}')
for a in range(F_sw.dim):
    for b in range(F_sw.dim):
        val = sp.simplify(sp.trigsimp(G_torsion[a, b]))
        if val != 0:
            print(f'  G[{a},{b}] = {val}')
print(f'  ε=0 recovers LC? {all(sp.simplify(G_torsion[a,b].subs(epsilon, 0) - einstein_tensor(LC, g_sw)[a,b]) == 0 for a in range(4) for b in range(4))}')

Pattern 3: Schwarzschild + ε torsion
  deformed vacuum? False
  G[0,3] = epsilon*(-2*M + r)*sin(theta)**2
  G[3,0] = epsilon*(2*M - r)*sin(theta)**2
  ε=0 recovers LC? True


#### Pattern 4: Schwarzschild + Weyl non-metricity (4D)

Same Weyl construction, on Schwarzschild. **Use `optimized=True`** because the 4D pipeline is much slower without it.

In [42]:
W_r = sp.Symbol('W_r', real=True)
LC = levi_civita(g_sw, optimized=True)   # optimized!
g_mat = g_sw.matrix()
g_inv = g_mat.inv()
W = [0, W_r, 0, 0]
W_up = [sum(g_inv[mu, nu] * W[nu] for nu in range(F_sw.dim)) for mu in range(F_sw.dim)]

Gamma = sp.MutableDenseNDimArray(LC.components)
for mu in range(F_sw.dim):
    for nu in range(F_sw.dim):
        for rho in range(F_sw.dim):
            d_munu = 1 if mu == nu else 0
            d_murho = 1 if mu == rho else 0
            Gamma[mu, nu, rho] += sp.Rational(1, 2) * (
                d_munu * W[rho] + d_murho * W[nu] - g_mat[nu, rho] * W_up[mu]
            )

weyl_conn = ComponentConnection(F_sw, Gamma)
G_weyl = einstein_tensor(weyl_conn, g_sw, optimized=True)
print('Pattern 4: Schwarzschild + W_r Weyl')
print(f'  deformed vacuum? {G_weyl.is_vacuum()}')
for a in range(F_sw.dim):
    for b in range(F_sw.dim):
        val = sp.simplify(sp.trigsimp(G_weyl[a, b]))
        if val != 0:
            print(f'  G[{a},{b}] = {val}')

Pattern 4: Schwarzschild + W_r Weyl
  deformed vacuum? False
  G[0,0] = W_r*(4*M**2*W_r*r + 24*M**2 - 4*M*W_r*r**2 - 28*M*r + W_r*r**3 + 8*r**2)/(4*r**3)
  G[1,1] = W_r*(6*M*W_r*r + 12*M - 3*W_r*r**2 - 8*r)/(4*r*(-2*M + r))
  G[2,2] = W_r*r*(2*M*W_r - W_r*r - 4)/4
  G[3,3] = W_r*r*(2*M*W_r - W_r*r - 4)*sin(theta)**2/4


#### Pattern 5: FLRW + scalar-gradient projective deformation

Connection deformed by a scalar field's gradient — typical scalar-tensor gravity setup. Coupling form `Γ + δ A_a + δ A_a` where `A_μ = ∂_μ φ`.

In [43]:
tt, rr, thh, ph = sp.symbols('t r theta phi')
a_func = sp.Function('a')(tt)
varphi = sp.Function('varphi')(tt)
F_flrw = CoordinateFrame([tt, rr, thh, ph])

g_flrw = ComponentMetric(F_flrw, sp.Matrix([
    [-1, 0, 0, 0],
    [0, a_func**2, 0, 0],
    [0, 0, a_func**2 * rr**2, 0],
    [0, 0, 0, a_func**2 * rr**2 * sp.sin(thh)**2],
]))
LC = levi_civita(g_flrw)

A = [sp.diff(varphi, c) for c in F_flrw.coords]
Gamma = sp.MutableDenseNDimArray(LC.components)
for mu in range(F_flrw.dim):
    for nu in range(F_flrw.dim):
        for rho in range(F_flrw.dim):
            d_munu = 1 if mu == nu else 0
            d_murho = 1 if mu == rho else 0
            Gamma[mu, nu, rho] += d_munu * A[rho] + d_murho * A[nu]

scalar_conn = ComponentConnection(F_flrw, Gamma)
G_def = einstein_tensor(scalar_conn, g_flrw)
print('Pattern 5: FLRW + scalar-gradient')
print(f'  LC G_tt = {sp.simplify(einstein_tensor(LC, g_flrw)[0, 0])}')
print(f'  G_def[0,0] = {sp.simplify(G_def[0, 0])}')

Pattern 5: FLRW + scalar-gradient
  LC G_tt = -3*Derivative(a(t), t)**2/a(t)**2
  G_def[0,0] = 3*((-Derivative(varphi(t), t)**2 + Derivative(varphi(t), (t, 2)))*a(t)**2 - 3*a(t)*Derivative(a(t), t)*Derivative(varphi(t), t) - 2*Derivative(a(t), t)**2)/(2*a(t)**2)


### Insight summary

Five patterns, three structural lessons:

| Pattern | Recovers LC at | Lovelock 2D? |
|---|---|---|
| 2D + α torsion | `α = 0` | No (torsion breaks it) |
| 2D + W_x Weyl | `W_x = 0` | **Yes** (`G ≡ 0` always) |
| Schwarzschild + ε torsion | `ε = 0` | n/a (4D) |
| Schwarzschild + W_r Weyl | `W_r = 0` | n/a (4D) |
| FLRW + scalar-grad | `varphi'(t) = 0` | n/a (4D, non-vacuum) |

Key takeaways:

- **API stress-tests with arbitrary symbols** are valid and accidentally surface deep theorems (Lovelock 2D from the A, B examples).
- **Physically-meaningful examples** parameterise the deformation by a small number of fields/constants, with the `parameter → 0` limit recovering Levi-Civita.

## Summary

* `jacopy.frame_calc` is the component-level submodule for concrete metric calculations.
* Three frame types (CoordinateFrame, Tetrad, AbstractFrame) share a common `Frame` protocol; higher-level operations are frame-agnostic.
* Pipeline: `g → LC → R → Ric → R_scalar → G`. Default mode records full derivation traces; optimised mode skips them for Kerr-class performance.
* Library fixtures cover standard metrics; users build custom metrics on any frame.
* `derivation_chain(...)` lifts any per-entry trace to a `ProofChain` for paper-grade LaTeX rendering.
* SymPy is an opt-in dependency under `[components]`.

$$ds^2=-e^{2\phi(r)}c^2dt^2+e^{2\Lambda(r)}dr^2+r^2(d\theta^2+sin^2\theta d\varphi^2)$$

In [44]:
import sympy as sp
from jacopy.frame_calc import (
    CoordinateFrame, ComponentMetric, levi_civita, to_latex_table,
)

# --- 1) Koordinatlar + metric fonksiyonları ---
t, r, theta, phi = sp.symbols("t r θ φ", real=True)
c = sp.symbols("c", positive=True)
f_phi = sp.Function("φ")(r)
f_Lam = sp.Function("Λ")(r)

# --- 2) Metric matrisi (line element'ten) ---
g_mat = sp.Matrix([
    [-sp.exp(2*f_phi) * c**2,   0,                0,    0],
    [0,              sp.exp(2*f_Lam),    0,    0],
    [0,              0,                r**2, 0],
    [0,              0,                0,    r**2 * sp.sin(theta)**2],
])

# --- 3) Frame + ComponentMetric + Levi-Civita ---
F = CoordinateFrame((t, r, theta, phi))
g = ComponentMetric(F, sp.MutableDenseNDimArray(g_mat))
LC = levi_civita(g)

# --- 4) Tüm nonzero Christoffel sembollerini yazdır ---
coord_names = ["t", "r", "θ", "φ"]
i=0
print("Levi-Civita (Christoffel) coefficients — Γ^a_{bc} (nonzero, b ≤ c):\n")
for a in range(4):
    for b in range(4):
        for c_idx in range(b, 4):                    # alt-indeks simetri (b,c)
            val = sp.simplify(LC[a, b, c_idx])
            if val != 0:
                up   = coord_names[a]
                low1 = coord_names[b]
                low2 = coord_names[c_idx]
                print(f"  {i}  Γ^{up}_{{{low1}{low2}}}  =  {val}")
            i += 1




Levi-Civita (Christoffel) coefficients — Γ^a_{bc} (nonzero, b ≤ c):

  1  Γ^t_{tr}  =  Derivative(φ(r), r)
  10  Γ^r_{tt}  =  c**2*exp(-2*Λ(r) + 2*φ(r))*Derivative(φ(r), r)
  14  Γ^r_{rr}  =  Derivative(Λ(r), r)
  17  Γ^r_{θθ}  =  -r*exp(-2*Λ(r))
  19  Γ^r_{φφ}  =  -r*exp(-2*Λ(r))*sin(θ)**2
  25  Γ^θ_{rθ}  =  1/r
  29  Γ^θ_{φφ}  =  -sin(2*θ)/2
  36  Γ^φ_{rφ}  =  1/r
  38  Γ^φ_{θφ}  =  1/tan(θ)


In [45]:
from jacopy.frame_calc import curvature, ricci, ricci_scalar, einstein_tensor

# --- 6) Riemann → Ricci → R skalar → Einstein ---
Ric       = ricci(LC)
R_scalar  = ricci_scalar(LC, g)
G         = einstein_tensor(LC, g)

# Display kısaltması: φ', φ'', Λ', Λ''
phi_p, phi_pp = sp.Symbol("φ'"), sp.Symbol("φ''")
Lam_p, Lam_pp = sp.Symbol("Λ'"), sp.Symbol("Λ''")
pretty = {
    sp.diff(f_phi, r):       phi_p,
    sp.diff(f_phi, r, 2):    phi_pp,
    sp.diff(f_Lam, r):       Lam_p,
    sp.diff(f_Lam, r, 2):    Lam_pp,
}
i=0
# --- 7) Ricci tensor (nonzero, simetrik a ≤ b) ---
print("Ricci tensor — Ric_{ab} (nonzero, a ≤ b):\n")
for a in range(4):
    for b in range(a, 4):
        val = sp.simplify(Ric[a, b]).subs(pretty)
        if val != 0:
            print(f"  {i}  Ric_{{{coord_names[a]}{coord_names[b]}}}  =  {val}")
        i += 1

# --- 8) Ricci scalar ---
print("\nRicci scalar:")
print(f"  R = {sp.simplify(R_scalar).subs(pretty)}")

j=0
# --- 9) Einstein tensor ---
print("\nEinstein tensor — G_{ab} (nonzero, a ≤ b):\n")
for a in range(4):
    for b in range(a, 4):
        val = sp.simplify(G[a, b]).subs(pretty)
        if val != 0:
            print(f"  {j}  G_{{{coord_names[a]}{coord_names[b]}}}  =  {val}")
        j += 1



Ricci tensor — Ric_{ab} (nonzero, a ≤ b):

  0  Ric_{tt}  =  c**2*(r*Λ'*φ' - r*φ'**2 - r*φ'' - 2*φ')*exp(-2*Λ(r) + 2*φ(r))/r
  4  Ric_{rr}  =  -Λ'*φ' + φ'**2 + φ'' - 2*Λ'/r
  7  Ric_{θθ}  =  (-r*Λ' + r*φ' - exp(2*Λ(r)) + 1)*exp(-2*Λ(r))
  9  Ric_{φφ}  =  (-r*Λ' + r*φ' - exp(2*Λ(r)) + 1)*exp(-2*Λ(r))*sin(θ)**2

Ricci scalar:
  R = 2*(-r**2*Λ'*φ' + r**2*φ'**2 + r**2*φ'' - 2*r*Λ' + 2*r*φ' - exp(2*Λ(r)) + 1)*exp(-2*Λ(r))/r**2

Einstein tensor — G_{ab} (nonzero, a ≤ b):

  0  G_{tt}  =  c**2*(-2*r*Λ' - exp(2*Λ(r)) + 1)*exp(-2*Λ(r) + 2*φ(r))/r**2
  4  G_{rr}  =  (-2*r*φ' + exp(2*Λ(r)) - 1)/r**2
  7  G_{θθ}  =  r*(r*Λ'*φ' - r*φ'**2 - r*φ'' + Λ' - φ')*exp(-2*Λ(r))
  9  G_{φφ}  =  r*(r*Λ'*φ' - r*φ'**2 - r*φ'' + Λ' - φ')*exp(-2*Λ(r))*sin(θ)**2


In [ ]:
"""
Kerr metric — full pipeline:
  Christoffel (Γ) → Riemann → Ricci → R-scalar → Einstein,
all entries printed with index numbering, no parallelism (≈15s total).
"""

import sympy as sp
import time
from itertools import product
from jacopy.frame_calc import (
    CoordinateFrame, ComponentMetric, levi_civita,
)

# ============================================================== #
# 1) Coordinates + Kerr parameters                                #
# ============================================================== #

t, r, theta, varphi = sp.symbols("t r θ φ", real=True)
c = sp.symbols("c", positive=True)
M = sp.symbols("M", positive=True)
a_spin = sp.symbols("a", real=True)

Sigma = r**2 + a_spin**2 * sp.cos(theta)**2
Delta = r**2 - 2*M*r + a_spin**2

# ============================================================== #
# 2) Kerr metric (Boyer–Lindquist coordinates)                    #
# ============================================================== #

g_tt          = -(1 - 2*M*r/Sigma) * c**2
g_tphi        = -2*M*a_spin*r*c*sp.sin(theta)**2 / Sigma
g_rr          = Sigma / Delta
g_thetatheta  = Sigma
g_phiphi      = (
    r**2 + a_spin**2 + 2*M*a_spin**2*r*sp.sin(theta)**2 / Sigma
) * sp.sin(theta)**2

g_mat = sp.Matrix([
    [g_tt,    0,    0,             g_tphi  ],
    [0,       g_rr, 0,             0       ],
    [0,       0,    g_thetatheta,  0       ],
    [g_tphi,  0,    0,             g_phiphi],
])

F = CoordinateFrame((t, r, theta, varphi))
g = ComponentMetric(F, sp.MutableDenseNDimArray(g_mat))

coord_names = ["t", "r", "θ", "φ"]
coord_syms  = (t, r, theta, varphi)
n = 4

# ============================================================== #
# 3) Levi-Civita (Christoffel) — package call                     #
# ============================================================== #

print(f"[{time.strftime('%H:%M:%S')}] Computing Levi-Civita...", flush=True)
t0 = time.time()
LC = levi_civita(g, optimized=True)
print(f"[{time.strftime('%H:%M:%S')}] LC done ({time.time()-t0:.1f}s)", flush=True)

# ============================================================== #
# 4) Riemann (manual serial) — fast for Kerr                      #
# ============================================================== #

print(f"[{time.strftime('%H:%M:%S')}] Computing Riemann...", flush=True)
t0 = time.time()
LC_comp = [[[LC[a, b, c_] for c_ in range(n)] for b in range(n)] for a in range(n)]
R = sp.MutableDenseNDimArray.zeros(n, n, n, n)
for a, b, c_, d in product(range(n), repeat=4):
    term1 = sp.diff(LC_comp[a][c_][d], coord_syms[b])
    term2 = sp.diff(LC_comp[a][b][d], coord_syms[c_])
    prod1 = sum(LC_comp[a][b][e] * LC_comp[e][c_][d] for e in range(4))
    prod2 = sum(LC_comp[a][c_][e] * LC_comp[e][b][d] for e in range(4))
    R[a, b, c_, d] = term1 - term2 + prod1 - prod2
print(f"[{time.strftime('%H:%M:%S')}] Riemann done ({time.time()-t0:.1f}s)", flush=True)

# ============================================================== #
# 5) Ricci, R-scalar, Einstein                                    #
# ============================================================== #

print(f"[{time.strftime('%H:%M:%S')}] Computing Ric, R-scalar, G...", flush=True)
t0 = time.time()

Ric = sp.MutableDenseNDimArray.zeros(n, n)
for b in range(n):
    for d in range(n):
        Ric[b, d] = sum(R[a, b, a, d] for a in range(n))

g_inv_mat = sp.Matrix(g_mat).inv()
R_scalar  = sum(g_inv_mat[a, b] * Ric[a, b] for a in range(n) for b in range(n))

G = sp.MutableDenseNDimArray.zeros(n, n)
for a in range(n):
    for b in range(n):
        G[a, b] = Ric[a, b] - sp.Rational(1, 2) * g_mat[a, b] * R_scalar

print(f"[{time.strftime('%H:%M:%S')}] Assembled ({time.time()-t0:.1f}s)", flush=True)

# ============================================================== #
# 6) Simplify all entries                                         #
# ============================================================== #

print(f"[{time.strftime('%H:%M:%S')}] Simplifying all entries...", flush=True)
t0 = time.time()

LC_simp     = {(a, b, c_): sp.simplify(LC[a, b, c_]) for a, b, c_ in product(range(n), repeat=3)}
Ric_simp    = {(a, b):      sp.simplify(Ric[a, b])  for a, b   in product(range(n), repeat=2)}
G_simp      = {(a, b):      sp.simplify(G[a, b])    for a, b   in product(range(n), repeat=2)}
R_scalar_s  = sp.simplify(R_scalar)

print(f"[{time.strftime('%H:%M:%S')}] Simplify done ({time.time()-t0:.1f}s)\n", flush=True)

# ============================================================== #
# 7) PRINT — full output (every entry, with index)                #
# ============================================================== #

print("=" * 70)
print("Levi-Civita coefficients — Γ^a_{bc}:\n")
i = 0
for up in range(4):
    for lo1 in range(4):
        for lo2 in range(4):
            val = LC_simp[(up, lo1, lo2)]
            print(f"  {i:>3}  Γ^{coord_names[up]}_{{{coord_names[lo1]}{coord_names[lo2]}}}  =  {val}")
            i += 1

print("\n" + "=" * 70)
print("Ricci tensor — Ric_{ab}:\n")
i = 0
for row in range(4):
    for col in range(4):
        val = Ric_simp[(row, col)]
        print(f"  {i:>3}  Ric_{{{coord_names[row]}{coord_names[col]}}}  =  {val}")
        i += 1

print("\n" + "=" * 70)
print("Ricci scalar:\n")
print(f"  R = {R_scalar_s}")

print("\n" + "=" * 70)
print("Einstein tensor — G_{ab}:\n")
j = 0
for row in range(4):
    for col in range(4):
        val = G_simp[(row, col)]
        print(f"  {j:>3}  G_{{{coord_names[row]}{coord_names[col]}}}  =  {val}")
        j += 1

print("\n" + "=" * 70)
print(f"Vacuum check: all Ric entries zero? "
      f"{all(Ric_simp[(a, b)] == 0 for a, b in product(range(n), repeat=2))}")
